# Homework

## Dataset
For this homework, we'll use the 2026 Car Fuel Efficiency dataset. Download it from here.

You can do it with wget:

```wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv```

The goal of this homework is to create a regression model for predicting the car fuel efficiency (column 'fuel_efficiency_mpg').

## Preparing the dataset
Use only the following columns:

- ```'engine_displacement'```,
- ```'horsepower'```,
- ```'vehicle_weight'```,
- ```'model_year'```,
- ```'fuel_efficiency_mpg'```

## EDA
Look at the ```fuel_efficiency_mpg variable```. Does it have a long tail?

In [1]:
import os
import pandas as pd

filename = "car_fuel_efficiency_2026.csv"
url = "https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/car_fuel_efficiency_2026.csv"

if not os.path.exists(filename):
    !wget {url} -O {filename}

df = pd.read_csv(filename)
df.head()

,model_year,origin,fuel_type,drivetrain,num_doors,engine_displacement,num_cylinders,horsepower,vehicle_weight,acceleration,fuel_efficiency_mpg
0,2006,Europe,Gasoline,Front-wheel drive,4,2180,6,243.0,3870,NaN,31.9
1,2008,Europe,Diesel,Front-wheel drive,4,2390,6,272.0,4210,NaN,31.3
2,1996,Asia,Gasoline,Front-wheel drive,5,2320,6,267.0,4240,17.3,27.5
3,1989,Europe,Gasoline,Front-wheel drive,4,2130,6,258.0,4490,18.7,28.5
4,1994,USA,Diesel,Front-wheel drive,3,2580,7,304.0,4510,17.5,31.0


## Question 1
There's one column with missing values. What is it?

- 'engine_displacement'
- 'horsepower'
- 'vehicle_weight'
- 'model_year'

In [2]:
cols = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']
missing = df[cols].isnull().sum()

print(missing)

cols_with_missing = missing[missing > 0].index.tolist()
print(f"\nColumns with missing values ({len(cols_with_missing)}): {cols_with_missing}")

engine_displacement      0
horsepower             877
vehicle_weight           0
model_year               0
dtype: int64

Columns with missing values (1): ['horsepower']


## Question 2
What's the median (50% percentile) for variable ```'horsepower'```?

- 204
- 254
- 304
- 354

In [3]:
float(df['horsepower'].median())

254.0

### Prepare and split the dataset
Shuffle the filtered dataset and create the split exactly as in the lecture:

```
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]]
df_val = df.iloc[idx[n_train:n_train + n_val]]
df_test = df.iloc[idx[n_train + n_val:]]
```

For Q5, repeat the same block with each listed seed. For Q6, use seed ```9```.

## Question 3
- We need to deal with missing values for the column from Q1.
- We have two options: fill it with 0 or with the mean of this variable.
- Try both options. For each, train a linear regression model without regularization using the code from the lessons.
- For computing the mean, use the training only!
- Use the validation dataset to evaluate the models and compare the RMSE of each option.
- Round the RMSE scores to 3 decimal digits using ```round(score, 3)```. This keeps the imputation difference visible in this release.
- Which option gives better RMSE?

Options:

- With 0
- With mean
- Both are equally good

In [4]:
# Step 1: keep only the columns we need
base = ['engine_displacement', 'horsepower', 'vehicle_weight', 'model_year']
df = df[base + ['fuel_efficiency_mpg']]

In [7]:
# Step 2: shuffle and split 60/20/20 with seed 42
import numpy as np

n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(42)
idx = np.arange(n)
np.random.shuffle(idx)

df_train = df.iloc[idx[:n_train]].reset_index(drop=True)
df_val = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)
df_test = df.iloc[idx[n_train + n_val:]].reset_index(drop=True)

In [8]:
# Step 3: separate the target from the features
y_train = df_train['fuel_efficiency_mpg'].values
y_val = df_val['fuel_efficiency_mpg'].values
y_test = df_test['fuel_efficiency_mpg'].values

del df_train['fuel_efficiency_mpg']
del df_val['fuel_efficiency_mpg']
del df_test['fuel_efficiency_mpg']

In [9]:
# Step 4: functions (first two from the lecture, third adapted)
def train_linear_regression(X, y):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

def rmse(y, y_pred):
    se = (y - y_pred) ** 2
    mse = se.mean()
    return np.sqrt(mse)

def prepare_X(df, fill_value):
    df = df.copy()
    df['horsepower'] = df['horsepower'].fillna(fill_value)
    return df.values

In [12]:
# Step 5: option A, fill missing horsepower with 0
X_train = prepare_X(df_train, 0)
w0, w = train_linear_regression(X_train, y_train)

X_val = prepare_X(df_val, 0)
y_pred = w0 + X_val.dot(w)

score_zero = round(rmse(y_val, y_pred), 3)
print('With 0:   ', score_zero)

With 0:    2.205


In [13]:
# Step 6: option B, fill with the mean from the TRAINING set only
mean_hp = df_train['horsepower'].mean()

X_train = prepare_X(df_train, mean_hp)
w0, w = train_linear_regression(X_train, y_train)

X_val = prepare_X(df_val, mean_hp)
y_pred = w0 + X_val.dot(w)

score_mean = round(rmse(y_val, y_pred), 3)
print('With mean:', score_mean)

With mean: 2.202


In [15]:
# Step 7: pick the answer from the options
if score_zero < score_mean:
    answer = 'With 0'
elif score_mean < score_zero:
    answer = 'With mean'
else:
    answer = 'Both are equally good'

print('Which option gives better RMSE?', answer)

Which option gives better RMSE? With mean


## Question 4
Now let's train a regularized linear regression.
For this question, fill the NAs with 0.
Try different values of ```r``` from this list: ```[0, 0.01, 0.1, 1, 5, 10, 100]```.
Use RMSE to evaluate the model on the validation dataset.
Round the RMSE scores to 4 decimal digits. This keeps the small but real regularization differences visible instead of turning several choices into a tie.
Which ```r``` gives the best RMSE?
If multiple options give the same best RMSE, select the smallest ```r```.

Options:

- 0
- 0.01
- 0.1
- 1
- 5
- 10
- 100

In [16]:
# Step 1: regularised linear regression
def train_linear_regression_reg(X, y, r=0.001):
    ones = np.ones(X.shape[0])
    X = np.column_stack([ones, X])

    XTX = X.T.dot(X)
    XTX = XTX + r * np.eye(XTX.shape[0])

    XTX_inv = np.linalg.inv(XTX)
    w_full = XTX_inv.dot(X.T).dot(y)

    return w_full[0], w_full[1:]

In [17]:
# Step 2: try each r, filling missing horsepower with 0
scores = {}

for r in [0, 0.01, 0.1, 1, 5, 10, 100]:
    X_train = prepare_X(df_train, 0)
    w0, w = train_linear_regression_reg(X_train, y_train, r=r)

    X_val = prepare_X(df_val, 0)
    y_pred = w0 + X_val.dot(w)
    score = round(rmse(y_val, y_pred), 4)

    scores[r] = score
    print(r, score)

0 2.2053
0.01 2.2058
0.1 2.2241
1 2.3492
5 2.4094
10 2.4195
100 2.4292


In [18]:
# Step 3: pick the best r (smallest r if there's a tie)
best_score = min(scores.values())
best_r = min(r for r in scores if scores[r] == best_score)

print('Answer:', best_r)

Answer: 0


## Question 5
We used seed 42 for splitting the data. Let's find out how selecting the seed influences our score.
Try different seed values: ```[0, 1, 2, 3, 4, 5, 6, 7, 8, 9]```.
For each seed, do the train/validation/test split with 60%/20%/20% distribution.
Fill the missing values with 0 and train a model without regularization.
For each seed, evaluate the model on the validation dataset and collect the RMSE scores.
What's the standard deviation of all the scores? To compute the standard deviation, use ```np.std```.
Round the result to 3 decimal digits ```(round(std, 3))```

What's the value of std?

- 0.006
- 0.016
- 0.029
- 0.036

In [19]:
# Step 1: repeat the split, train and evaluate for each seed
rmse_scores = []

for seed in [0, 1, 2, 3, 4, 5, 6, 7, 8, 9]:
    # same split block as before, just with a different seed
    n = len(df)
    n_val = int(n * 0.2)
    n_test = int(n * 0.2)
    n_train = n - n_val - n_test

    np.random.seed(seed)
    idx = np.arange(n)
    np.random.shuffle(idx)

    df_train_s = df.iloc[idx[:n_train]].reset_index(drop=True)
    df_val_s = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)

    y_train_s = df_train_s['fuel_efficiency_mpg'].values
    y_val_s = df_val_s['fuel_efficiency_mpg'].values

    del df_train_s['fuel_efficiency_mpg']
    del df_val_s['fuel_efficiency_mpg']

    # fill with 0, train without regularisation
    X_train = prepare_X(df_train_s, 0)
    w0, w = train_linear_regression(X_train, y_train_s)

    X_val = prepare_X(df_val_s, 0)
    y_pred = w0 + X_val.dot(w)
    score = rmse(y_val_s, y_pred)

    rmse_scores.append(score)
    print(seed, round(score, 3))

0 2.239
1 2.202
2 2.163
3 2.204
4 2.202
5 2.246
6 2.27
7 2.191
8 2.217
9 2.207


In [20]:
# Step 2: standard deviation of the ten scores
std = round(np.std(rmse_scores), 3)
print('Answer:', std)

Answer: 0.029


## Question 6
Split the dataset like previously, use seed 9.
Combine train and validation datasets.
Fill the missing values with 0 and train a model with ```r=0.001```.
What's the RMSE on the test dataset?

Options:

- 0.236
- 2.236
- 22.10
- 221.0

In [21]:
# Step 1: split with seed 9
n = len(df)
n_val = int(n * 0.2)
n_test = int(n * 0.2)
n_train = n - n_val - n_test

np.random.seed(9)
idx = np.arange(n)
np.random.shuffle(idx)

df_train_9 = df.iloc[idx[:n_train]].reset_index(drop=True)
df_val_9 = df.iloc[idx[n_train:n_train + n_val]].reset_index(drop=True)
df_test_9 = df.iloc[idx[n_train + n_val:]].reset_index(drop=True)

y_train_9 = df_train_9['fuel_efficiency_mpg'].values
y_val_9 = df_val_9['fuel_efficiency_mpg'].values
y_test_9 = df_test_9['fuel_efficiency_mpg'].values

del df_train_9['fuel_efficiency_mpg']
del df_val_9['fuel_efficiency_mpg']
del df_test_9['fuel_efficiency_mpg']

In [22]:
# Step 2: combine train and validation
df_full_train = pd.concat([df_train_9, df_val_9])
df_full_train = df_full_train.reset_index(drop=True)

X_full_train = prepare_X(df_full_train, 0)
y_full_train = np.concatenate([y_train_9, y_val_9])

In [23]:
# Step 3: train with r=0.001
w0, w = train_linear_regression_reg(X_full_train, y_full_train, r=0.001)

In [24]:
# Step 4: evaluate on the test set
X_test = prepare_X(df_test_9, 0)
y_pred = w0 + X_test.dot(w)
score = rmse(y_test_9, y_pred)

print('Test RMSE:', round(score, 3))

Test RMSE: 2.236


In [26]:
# Step 5: match to the options
options = [0.236, 2.236, 22.10, 221.0]
answer = min(options, key=lambda o: abs(o - score))
print('What\'s the RMSE on the test dataset?', answer)

What's the RMSE on the test dataset? 2.236
